In [1]:
%matplotlib qt
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append(str(Path("../scripts").resolve()))
import analisis_ttl as ttl
from roi_status_selector import (
    apply_registry_to_processing_tables,
    build_master_roi_outputs,
    load_roi_status_registry,
    prepare_roi_processing_selection,
    save_roi_processing_outputs,
    select_and_update_roi_status,
)

base_dir = Path("/Users/gfernandezv/Documents/envs/Images_TTL_temp/data/Proc_data")
excel_path = base_dir / "TRPM3_imagenes.xlsx"


### Notebook de processing

Este notebook parte desde los archivos `*_preprocessed_long.csv` generados en `01_preprocessing.ipynb`.

Aquí no se recalculan `phase` ni `trend`: esas marcas ya vienen desde el preprocesamiento. Este segundo paso se usa para:

- cargar las muestras activas del Excel;
- filtrar por genotipo, fase o tendencia;
- graficar y guardar tablas para análisis batch.

**ROI_status:** por defecto (`use_registry_overrides=False`) el `ROI_status` de cada ROI es exactamente el que quedó fijado en `01_preprocessing.ipynb` (inspección visual + `mark_roi_status`). El selector interactivo con lazo sigue disponible más abajo (`launch_roi_selector`), pero es opcional: solo pisa el `ROI_status` de 01 si además pones `use_registry_overrides=True`. Esto evita que una selección accidental en el lazo (o un registry acumulado de sesiones viejas) sobreescriba silenciosamente las decisiones manuales tomadas en 01.


In [ ]:
imports = ttl.load_exported_experiments(base_dir)

data_resume = imports["data_resume"]
active_experiments = imports["active_experiments"]
load_status = imports["load_status"]
preprocessed_all = imports["preprocessed_all"]

print(f"experimentos estado A: {len(active_experiments)}")
print(f"preprocessed_all: {preprocessed_all.shape}")

if preprocessed_all.empty:
    print("No se encontraron archivos *_preprocessed_long.csv. Ejecuta primero 01_preprocessing.ipynb para cada muestra.")

# Cada carpeta activa debe tener un unico *_preprocessed_long.csv. Mas de uno
# generalmente significa que el nombre de sample cambio entre corridas (por
# ejemplo por un cambio en la deteccion automatica de sample/genotype en
# 01_preprocessing.ipynb) y quedo un CSV viejo sin borrar: load_exported_experiments
# los concatena a todos, duplicando ROIs de esa muestra en preprocessed_all.
duplicated_preprocessed = load_status[load_status["preprocessed_files"].apply(len) > 1]
if not duplicated_preprocessed.empty:
    print("ADVERTENCIA: carpetas con mas de un *_preprocessed_long.csv (revisar y dejar solo el correcto):")
    print(duplicated_preprocessed[["folder", "preprocessed_files"]].to_string(index=False))

active_experiments[["folder", "Genotype", "nick name", "estado", "ROI frames"]]
load_status

In [ ]:
# Validacion: verifica que temp_ranges (low/mid/high, usado para clasificar
# trend) y norm_temp_range (ventana de corte pasada a process_sample() antes
# de normalizar, distinta de temp_ranges) sean los mismos para todas las
# muestras activas. Ambos quedan fijos desde 01_preprocessing.ipynb y no se
# recalculan aqui: si dos muestras se preprocesaron con valores distintos
# (por ejemplo temp_range=(20, 40) en una muestra y otro rango en otra),
# quedarian mezcladas bajo las mismas columnas en roi_temp_summary_active.csv
# sin ninguna advertencia.

def _check_range_consistency(df, cols, label):
    if df.empty:
        return
    if not cols:
        print(
            f"Advertencia: preprocessed_all no tiene columnas {label} "
            "(CSV generado antes de registrar esta metadata). "
            "No se puede validar consistencia entre muestras."
        )
        return

    by_file = df.groupby("source_file")[cols].first()
    inconsistent = [col for col in cols if by_file[col].nunique(dropna=False) > 1]

    if inconsistent:
        print(f"ADVERTENCIA: {label} no coincide entre todas las muestras activas.")
        for col in inconsistent:
            print(f"  {col} por source_file:")
            print(by_file[col].to_string())
    else:
        print(f"{label} usado en 01_preprocessing.ipynb (igual en todas las muestras activas):")
        print(by_file.iloc[0].to_dict())

temp_range_cols = [c for c in preprocessed_all.columns if c.startswith("temp_range_")]
norm_temp_range_cols = [c for c in preprocessed_all.columns if c.startswith("norm_temp_range_")]

_check_range_consistency(preprocessed_all, temp_range_cols, "temp_ranges (low/mid/high)")
print()
_check_range_consistency(
    preprocessed_all,
    norm_temp_range_cols,
    "norm_temp_range (ventana de corte antes de normalizar)",
)


In [5]:
genotype_filter = ["m65"]
phase_filter = "cooling"
trend_filter = "stable"
# Ejemplos:
# genotype_filter = None
# genotype_filter = ["m27", "m36"]
# phase_filter = None
# trend_filter = "increase"  # "increase", "decrease", "stable", "insufficient", None

In [ ]:
# Script de procesamiento.
roi_status_id_cols = ["source_folder", "source_file", "sample", "genotype", "genotype_meta", "nickname_meta", "ROI"]
roi_id_cols = roi_status_id_cols
batch_dir = base_dir / "batch_analysis"
batch_dir.mkdir(exist_ok=True)

# use_registry_overrides=False (default): ROI_status se toma tal cual viene de
# *_preprocessed_long.csv (la decision manual hecha en 01_preprocessing.ipynb),
# sin pasar por roi_status_registry.csv. Es la fuente de verdad mientras no se
# use el selector interactivo mas abajo.
# use_registry_overrides=True: aplica encima el registry acumulado de sesiones
# previas del selector interactivo (roi_status_registry.csv), que puede pisar
# el ROI_status de 01 para los ROI que ya fueron tocados con el selector.
use_registry_overrides = False

if use_registry_overrides:
    if "roi_status_registry" not in globals():
        roi_status_registry, roi_status_registry_path = load_roi_status_registry(
            batch_dir,
            id_cols=roi_status_id_cols,
        )
    elif "roi_status_registry_path" not in globals():
        roi_status_registry_path = batch_dir / "roi_status_registry.csv"
else:
    roi_status_registry = pd.DataFrame(columns=roi_status_id_cols + ["ROI_status"])
    roi_status_registry_path = batch_dir / "roi_status_registry.csv"

available_genotypes = sorted(preprocessed_all["genotype_meta"].dropna().astype(str).unique()) if not preprocessed_all.empty else []
print("Genotipos disponibles:", available_genotypes)
print("Usando ROI_status de 01_preprocessing.ipynb directamente" if not use_registry_overrides else f"ROIs en registry acumulado: {roi_status_registry.shape[0]}")
if use_registry_overrides and not roi_status_registry.empty:
    print(roi_status_registry["ROI_status"].value_counts().sort_index())

# temp_ranges_display es solo para presentacion/etiquetas en range_long y en
# los graficos de este notebook (Graph 1, group_summary). NO recalcula
# low_mean/mid_mean/high_mean ni trend: esos valores ya vienen fijos desde
# el CSV *_preprocessed_long.csv exportado por 01_preprocessing.ipynb. Debe
# coincidir con los temp_ranges usados en 01_preprocessing.ipynb para la(s)
# muestra(s) filtradas; la celda de validacion de temp_ranges mas arriba avisa
# si no coinciden entre muestras activas.
temp_ranges_display = {
    "low": (20, 27),
    "mid": (28, 33),
    "high": (34, 40),
}

# Aviso si temp_ranges_display quedo desincronizado con lo que realmente
# exporto 01_preprocessing.ipynb para las muestras que va a incluir este filtro.
_filtered_preview = ttl.filter_by_values(preprocessed_all, "genotype_meta", genotype_filter)
_filtered_preview = ttl.filter_by_values(_filtered_preview, "phase", phase_filter)
for _name, (_lo, _hi) in temp_ranges_display.items():
    _min_col, _max_col = f"temp_range_{_name}_min", f"temp_range_{_name}_max"
    if _min_col in _filtered_preview.columns and _max_col in _filtered_preview.columns:
        _actual = _filtered_preview[[_min_col, _max_col]].drop_duplicates()
        if not (_actual[_min_col].eq(_lo).all() and _actual[_max_col].eq(_hi).all()):
            print(
                f"ADVERTENCIA: temp_ranges_display['{_name}']={(_lo, _hi)} no coincide con "
                f"temp_range_{_name}_min/max real en preprocessed_all para el filtro actual: "
                f"{_actual.drop_duplicates().to_dict('records')}"
            )

processing_tables = prepare_roi_processing_selection(
    preprocessed_all,
    registry=roi_status_registry,
    id_cols=roi_status_id_cols,
    filter_func=ttl.filter_by_values,
    temp_summary_to_long_func=ttl.temp_summary_to_long,
    genotype_filter=genotype_filter,
    phase_filter=phase_filter,
    trend_filter=trend_filter,
    temp_ranges=temp_ranges_display,
)

processing_selected = processing_tables["processing_selected"]
processing_active = processing_tables["processing_active"]
roi_temp_summary = processing_tables["roi_temp_summary"]
roi_temp_summary_active = processing_tables["roi_temp_summary_active"]
range_long = processing_tables["range_long"]

available_rois_after_filters = sorted(processing_selected["ROI"].dropna().astype(str).unique()) if not processing_selected.empty else []

print("Filtro genotype:", "todos" if genotype_filter is None else genotype_filter)
print("Filtro phase:", "todas" if phase_filter is None else phase_filter)
print("Filtro trend:", "todos" if trend_filter is None else trend_filter)
print(f"processing_selected total: {processing_selected.shape}")
print(f"processing_active ROI_status=1: {processing_active.shape}")
print(f"ROIs disponibles después de filtros: {len(available_rois_after_filters)}")
print("ROI_status en filtro actual:")
print(processing_selected["ROI_status"].value_counts().sort_index())
print(available_rois_after_filters)
print("ROIs resumidas total:", roi_temp_summary.shape[0])
print("ROIs activas ROI_status=1:", roi_temp_summary_active.shape[0])
print("Puntos rango-ROI activos:", range_long.shape[0])
if not roi_temp_summary_active.empty:
    print(roi_temp_summary_active["trend"].value_counts())

roi_temp_summary.head()


In [ ]:
# El selector interactivo es opcional. Si lo usas, sus marcas se guardan en
# roi_status_registry.csv, pero solo se aplicaran a los datos si arriba
# pusiste use_registry_overrides=True (si no, el ROI_status de 01 sigue
# mandando en processing_tables aunque guardes marcas aqui).
launch_roi_selector = False

if launch_roi_selector:
    roi_temp_summary, roi_status_registry = select_and_update_roi_status(
        roi_temp_summary,
        registry=roi_status_registry,
        id_cols=roi_id_cols,
        registry_path=roi_status_registry_path,
        save_path=batch_dir / "roi_temp_summary_with_roi_status.csv",
    )

    if use_registry_overrides:
        processing_tables = apply_registry_to_processing_tables(
            processing_tables,
            registry=roi_status_registry,
            id_cols=roi_id_cols,
            temp_summary_to_long_func=ttl.temp_summary_to_long,
            temp_ranges=temp_ranges_display,
        )

        processing_selected = processing_tables["processing_selected"]
        processing_active = processing_tables["processing_active"]
        roi_temp_summary = processing_tables["roi_temp_summary"]
        roi_temp_summary_active = processing_tables["roi_temp_summary_active"]
        range_long = processing_tables["range_long"]
    else:
        print("use_registry_overrides=False: las marcas del selector quedaron guardadas en el registry, pero processing_tables sigue usando el ROI_status de 01_preprocessing.ipynb.")

print("ROI_status en resumen:")
print(roi_temp_summary["ROI_status"].value_counts().sort_index())


In [ ]:
#Graph 1
x_pos = {"low": 0, "mid": 1, "high": 2}
x_labels = [f"{name}\n{lo}-{hi}°C" for name, (lo, hi) in temp_ranges_display.items()]
label_roi_points = True
samples = sorted(range_long["sample"].dropna().unique()) if not range_long.empty else []
colors = dict(zip(samples, plt.cm.tab10(range(len(samples)))))

plt.figure(figsize=(8, 5))

for sample_name, sub in range_long.groupby("sample", dropna=False):
    xs = sub["temp_range"].map(x_pos).astype(float)
    jitter = ((sub.groupby("temp_range").cumcount() % 9) - 4) * 0.012
    plot_x = xs + jitter
    plt.scatter(
        plot_x,
        sub["mean_normsignal"],
        s=22,
        alpha=0.45,
        color=colors.get(sample_name, "0.5"),
        label=sample_name,
    )

    if label_roi_points:
        for x, y, roi in zip(plot_x, sub["mean_normsignal"], sub["ROI"]):
            plt.annotate(
                str(roi),
                (x, y),
                xytext=(3, 3),
                textcoords="offset points",
                fontsize=7,
                alpha=0.75,
            )

if not range_long.empty:
    group_summary = range_long.groupby("temp_range", observed=True).agg(
        mean=("mean_normsignal", "mean"),
        sem=("mean_normsignal", lambda x: x.std() / (len(x) ** 0.5)),
        n=("mean_normsignal", "size"),
    )
    group_x = [x_pos[idx] for idx in group_summary.index]
    plt.errorbar(
        group_x,
        group_summary["mean"],
        yerr=group_summary["sem"],
        color="black",
        marker="o",
        linewidth=2.5,
        capsize=4,
        label="mean ± SEM",
    )
else:
    group_summary = pd.DataFrame()

plt.axhline(0, linestyle="--", alpha=0.4)
plt.xticks([0, 1, 2], x_labels)
plt.ylabel("Mean NormSignal per ROI")
plt.xlabel("Temperature range")
plt.title("Active ROIs: response by temperature range")
plt.legend(title="Sample", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

group_summary


In [ ]:
#Graph2
graph2_samples = sorted(roi_temp_summary_active["sample"].dropna().unique()) if not roi_temp_summary_active.empty else []
graph2_colors = dict(zip(graph2_samples, plt.cm.tab10(range(len(graph2_samples)))))

plt.figure(figsize=(7, 4))
for sample_name, sub in roi_temp_summary_active.groupby("sample", dropna=False):
    plt.scatter(
        sub["low_mean"],
        sub["high_mean"],
        s=28,
        alpha=0.55,
        color=graph2_colors.get(sample_name, "0.5"),
        label=sample_name,
    )

if not roi_temp_summary_active.empty:
    lims = [
        min(roi_temp_summary_active["low_mean"].min(), roi_temp_summary_active["high_mean"].min()),
        max(roi_temp_summary_active["low_mean"].max(), roi_temp_summary_active["high_mean"].max()),
    ]
    plt.plot(lims, lims, color="black", linestyle="--", alpha=0.5)

plt.xlabel("Low mean NormSignal")
plt.ylabel("High mean NormSignal")
plt.title("Active ROIs: high vs low temperature response")
plt.legend(title="Sample", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

roi_temp_summary_active.groupby(["sample", "trend"]).size().unstack(fill_value=0)

In [ ]:
# Guarda tablas del processing en data/Proc_data/batch_analysis
# Archivo principal para análisis:
# - roi_temp_summary_active.csv contiene TODAS las ROI activas acumuladas.
# Archivos auxiliares:
# - *_current_filter_* conservan la vista del filtro actual.
# - *_all_* conservan el universo completo con ROI_status acumulado (segun
#   use_registry_overrides: por defecto, directo desde 01_preprocessing.ipynb).
save_batch_outputs = True

if save_batch_outputs:
    if use_registry_overrides:
        roi_status_registry.to_csv(roi_status_registry_path, index=False)
        print(f"Registry acumulado guardado: {roi_status_registry_path}")

    master_tables = build_master_roi_outputs(
        preprocessed_all,
        registry=roi_status_registry,
        id_cols=roi_status_id_cols,
        temp_summary_to_long_func=ttl.temp_summary_to_long,
        temp_ranges=temp_ranges_display,
    )

    saved_paths = save_roi_processing_outputs(
        current_tables=processing_tables,
        master_tables=master_tables,
        save_func=ttl.save_batch_dataframe,
        base_dir=base_dir,
    )
